In [ ]:
import torch
import time
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoConfig, AutoModel, ModernBertConfig, ModernBertModel
import transformers
import mamba_ssm

In [ ]:
print(f"transformers version: {transformers.__version__}")
print(f"mamba_ssm version: {mamba_ssm.__version__}")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
param_count = lambda m: sum(p.numel() for p in m.parameters())
print("Initializing Caduceus...")
model_name = "kuleshov-group/compo-cad2-l48-d1536-dna-chtk-c8k-1t-v1-b2-lr4e4-NzqiLr"
plantcad2 = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)

In [ ]:
print(f"Number of Caduceus parameters: {param_count(plantcad2)}")
print(f"d_model: {plantcad2.config.d_model}, n_layer: {plantcad2.config.n_layer}")

In [ ]:
# --- Setup ---
device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("Warning: Running on CPU. Performance benchmarks will not be representative.")

# --- Model 1: Caduceus (Pretrained) ---
print("Initializing Caduceus...")
model_name = "kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16"
d_model, n_layer = 1536*2, 48

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True, d_model=d_model//2, n_layer=n_layer)
caduceus = AutoModel.from_config(config, trust_remote_code=True).to(device)

# --- Model 2: ModernBert (From Scratch) ---
print("Initializing ModernBert...")
bert = ModernBertModel(ModernBertConfig(
                    vocab_size=config.vocab_size,
                    hidden_size=d_model,
                    intermediate_size=d_model * 4,
                    num_hidden_layers=n_layer,
                    num_attention_heads=8,
                    pad_token_id=tokenizer.convert_tokens_to_ids("[PAD]"),
                )).to(device)

param_count = lambda m: sum(p.numel() for p in m.parameters())
print(f"Number of Caduceus parameters: {param_count(caduceus)}")
print(f"Number of ModernBERT parameters: {param_count(bert)}")

In [ ]:
models_to_test = {
    "Caduceus": caduceus,
    "ModernBert": bert
}

In [ ]:
# --- Benchmark Configuration ---
# highlight-start
SEQUENCE_LENGTH = 81920
# A long sequence length will use a lot of memory, so we test small batch sizes.
BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64]
# highlight-end

results = []
warmup_runs = 2
inference_runs = 5

print(f"\nStarting benchmark on device: {device}")
# highlight-start
print(f"Fixed Sequence Length: {SEQUENCE_LENGTH}")
# highlight-end
print(f"Warmup runs: {warmup_runs}, Inference runs per test: {inference_runs}\n")

# highlight-start
for batch_size in BATCH_SIZES:
    print(f"--- Testing Batch Size: {batch_size} ---")
    
    # Create a batch of identical DNA sequences
    dna_sequences = [''.join(random.choices('ACGT', k=SEQUENCE_LENGTH))] * batch_size
    
    # The tokenizer handles a list of sequences automatically to create a batch
    input_ids = tokenizer(dna_sequences, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
# highlight-end

    for model_name, model in models_to_test.items():
        try:
            # Memory measurement
            torch.cuda.reset_peak_memory_stats(device)
            
            # Speed measurement (with warm-up)
            with torch.inference_mode():
                for _ in range(warmup_runs):
                    _ = model(input_ids)

                start_time = time.monotonic()
                for _ in range(inference_runs):
                    _ = model(input_ids)
                end_time = time.monotonic()

            peak_memory_gb = torch.cuda.max_memory_allocated(device) / (1024**3)
            avg_time_ms = ((end_time - start_time) / inference_runs) * 1000

            results.append({
                "Model": model_name,
                # highlight-start
                "Batch Size": batch_size,
                # highlight-end
                "Inference Time (ms)": avg_time_ms,
                "Peak Memory (GB)": peak_memory_gb
            })
            print(f"  {model_name}: {avg_time_ms:.2f} ms, {peak_memory_gb:.3f} GB")

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  {model_name}: Out of Memory")
                results.append({
                    "Model": model_name,
                    # highlight-start
                    "Batch Size": batch_size,
                    # highlight-end
                    "Inference Time (ms)": np.nan,
                    "Peak Memory (GB)": np.nan
                })
                # If a model fails, no point trying it with larger batches
                break 
            else:
                raise e

# --- Display Results ---
print("\n--- Benchmark Summary ---")
df = pd.DataFrame(results)
# Pivot the table for easier comparison across batch sizes
try:
    df_pivot = df.pivot(index='Batch Size', columns='Model', values=['Inference Time (ms)', 'Peak Memory (GB)'])
    print(df_pivot.to_string())
except Exception as e:
    print("Could not pivot DataFrame, printing as is:")
    print(df.to_string())